# Build Spark Session

In [2]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DataRead")
    .master("local[*]")
    .getOrCreate()
)

spark

# Cocatinated by Name for Each Department

In [4]:
from pyspark.sql import functions as F


employer_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Aggregations\employee_salary_data.csv")
)
print("Source Data ")
employer_df.show()

print("Concatenating Employee Name by Department")
employer_df = (
    employer_df.groupBy("Department")
    .agg(F.concat_ws(",", F.collect_list("employee_name")).alias("Employee Names"))
).show(truncate=False)




Source Data 
+-----------+-------------+-----------+-------+
|employee_id|employee_name| department| salary|
+-----------+-------------+-----------+-------+
|          1|        Aarav|Engineering|1250000|
|          2|        Meera|Engineering| 780000|
|          3|        Rohan|Engineering| 920000|
|          4|        Nisha|         HR| 680000|
|          5|        Priya|         HR| 740000|
|          6|       Vikram|         HR| 850000|
|          7|         Neha|    Support| 620000|
|          8|       Aditya|    Support|1100000|
|          9|        Kavya|    Support| 560000|
|         10|        Imran|    Support| 980000|
+-----------+-------------+-----------+-------+

Concatenating Employee Name by Department
+-----------+-----------------------+
|Department |Employee Names         |
+-----------+-----------------------+
|Engineering|Aarav,Meera,Rohan      |
|HR         |Nisha,Priya,Vikram     |
|Support    |Neha,Aditya,Kavya,Imran|
+-----------+-----------------------+



### Rows into Columns

In [6]:
products_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Transformations\Products_MultiColumn.csv")
)
print("Source Data ")
products_df.show()

print("Target Data into Single Column")
products_df.select(products_df.PRODUCT_ID1.alias("PRODUCT_ID"))\
           .union(products_df.select(products_df.PRODUCT_ID2.alias("PRODUCT_ID")))\
           .union(products_df.select(products_df.PRODUCT_ID3.alias("PRODUCT_ID")))\
           .orderBy("PRODUCT_ID")\
           .show(truncate=False)

Source Data 
+-----------+-----------+-----------+
|PRODUCT_ID1|PRODUCT_ID2|PRODUCT_ID3|
+-----------+-----------+-----------+
|    P000001|    P000002|    P000003|
|    P000004|    P000005|    P000006|
+-----------+-----------+-----------+

Target Data into Single Column
+----------+
|PRODUCT_ID|
+----------+
|P000001   |
|P000002   |
|P000003   |
|P000004   |
|P000005   |
|P000006   |
+----------+



### Columns into Rows

In [9]:
products_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Transformations\Products_Sku.csv")
)
print("Source Data ")
products_df.show()

print("Target Data ")

from pyspark.sql.functions import collect_list, sort_array, col

output_df = (
    products_df.groupBy("PRODUCT_ID")
      .agg(sort_array(collect_list("SKU_ID")).alias("SKU_LIST"))
      .select(
          col("PRODUCT_ID"),
          col("SKU_LIST")[0].alias("PRIMARY_SKU_ID"),
          col("SKU_LIST")[1].alias("SECONDARY_SKU_ID"),
          col("SKU_LIST")[2].alias("OTHER_SKU_ID")
      )
).show(truncate=False)

Source Data 
+----------+------+
|PRODUCT_ID|SKU_ID|
+----------+------+
|   P000001|    AA|
|   P000001|    BB|
|   P000001|    CC|
|   P000002|    AA|
|   P000002|    BB|
|   P000002|    CC|
+----------+------+

Target Data 
+----------+--------------+----------------+------------+
|PRODUCT_ID|PRIMARY_SKU_ID|SECONDARY_SKU_ID|OTHER_SKU_ID|
+----------+--------------+----------------+------------+
|P000001   |AA            |BB              |CC          |
|P000002   |AA            |BB              |CC          |
+----------+--------------+----------------+------------+



### Combine all below three tables based on column names

In [11]:
poll1_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Transformations\Sequence_Poll_1.csv")
)

poll2_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Transformations\Sequence_Poll_2.csv")
)

poll3_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Transformations\Sequence_Poll_3.csv")
)
print("Source Data - Poll 1")
poll1_df.show()

print("Source Data - Poll 2")
poll2_df.show()

print("Source Data - Poll 3")
poll3_df.show()

print("Target Data - Union of Polls")
poll1_df.unionByName(poll2_df,allowMissingColumns=True).unionByName(poll3_df,allowMissingColumns=True).show(truncate=False)

Source Data - Poll 1
+----+----+----+----+
|SEQ1|SEQ2|SEQ3|SEQ4|
+----+----+----+----+
|   1|   2|   3|   4|
+----+----+----+----+

Source Data - Poll 2
+----+----+----+----+
|SEQ4|SEQ3|SEQ2|SEQ1|
+----+----+----+----+
|   8|   7|   6|   5|
+----+----+----+----+

Source Data - Poll 3
+----+----+
|SEQ1|SEQ4|
+----+----+
|   9|  10|
+----+----+

Target Data - Union of Polls
+----+----+----+----+
|SEQ1|SEQ2|SEQ3|SEQ4|
+----+----+----+----+
|1   |2   |3   |4   |
|5   |6   |7   |8   |
|9   |NULL|NULL|10  |
+----+----+----+----+



### Get Processing Source File Name

In [7]:
from pyspark.sql.functions import *

poll1_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Transformations\Sequence_Poll_1.csv")
    .withColumn(
        "InputFile_Name",
        element_at(split(input_file_name(), "/"), -1))
)

poll1_df.show(truncate=False)

+----+----+----+----+-------------------+
|SEQ1|SEQ2|SEQ3|SEQ4|InputFile_Name     |
+----+----+----+----+-------------------+
|1   |2   |3   |4   |Sequence_Poll_1.csv|
+----+----+----+----+-------------------+



### lead and lag

In [8]:
from pyspark.sql.functions import *
from pyspark.sql.window import *

employer_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Aggregations\employee_salary_data.csv")
    .withColumn("Lag_1_Row_Salary", lag("Salary", 1).over(Window.partitionBy("Department").orderBy("Employee_ID")))
    .withColumn("Lag_2_Row_Salary", lag("Salary", 2).over(Window.partitionBy("Department").orderBy("Employee_ID")))
    .withColumn("Lead_1_Row_Salary", lead("Salary", 1).over(Window.partitionBy("Department").orderBy("Employee_ID")))
    .withColumn("Lead_2_Row_Salary", lead("Salary", 2).over(Window.partitionBy("Department").orderBy("Employee_ID")))
).show(truncate=False)

+-----------+-------------+-----------+-------+----------------+----------------+-----------------+-----------------+
|employee_id|employee_name|department |salary |Lag_1_Row_Salary|Lag_2_Row_Salary|Lead_1_Row_Salary|Lead_2_Row_Salary|
+-----------+-------------+-----------+-------+----------------+----------------+-----------------+-----------------+
|1          |Aarav        |Engineering|1250000|NULL            |NULL            |780000           |920000           |
|2          |Meera        |Engineering|780000 |1250000         |NULL            |920000           |NULL             |
|3          |Rohan        |Engineering|920000 |780000          |1250000         |NULL             |NULL             |
|4          |Nisha        |HR         |680000 |NULL            |NULL            |740000           |850000           |
|5          |Priya        |HR         |740000 |680000          |NULL            |850000           |NULL             |
|6          |Vikram       |HR         |850000 |740000   

### Greatest and least

In [11]:
from pyspark.sql.functions import *

mulproduct_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Transformations\Products_Multi_Price.csv")
)
print("Source Data ")
mulproduct_df.show()

print("Target Data ")
mulproduct_df.select(
    col("PROD_ID").alias("PRODUCT_ID"),
    greatest("PRICE_2017", "PRICE_2018", "PRICE_2019", "PRICE_2020").alias("GREATEST_PRICE"),
    least("PRICE_2017", "PRICE_2018", "PRICE_2019", "PRICE_2020").alias("LEAST_PRICE")
).show(truncate=False)

Source Data 
+-------+----------+----------+----------+----------+
|PROD_ID|PRICE_2017|PRICE_2018|PRICE_2019|PRICE_2020|
+-------+----------+----------+----------+----------+
| PR0001|       100|       200|       300|       400|
| PR0002|        50|       200|       300|       100|
+-------+----------+----------+----------+----------+

Target Data 
+----------+--------------+-----------+
|PRODUCT_ID|GREATEST_PRICE|LEAST_PRICE|
+----------+--------------+-----------+
|PR0001    |400           |100        |
|PR0002    |300           |50         |
+----------+--------------+-----------+



### Horizental Addition of Column Values

In [12]:
from pyspark.sql.functions import *

mulproduct_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Transformations\Products_Multi_Price.csv")
)
print("Source Data ")
mulproduct_df.show()

print("Target Data ")
mulproduct_df.select(
    col("PROD_ID").alias("PRODUCT_ID"),
    repeat("PRICE_2017", 3).alias("REPEAT_PRICE_2017"),
    repeat("PRICE_2018", 3).alias("REPEAT_PRICE_2018"),
    repeat("PRICE_2019", 3).alias("REPEAT_PRICE_2019"),
    repeat("PRICE_2020", 3).alias("REPEAT_PRICE_2020")
).show(truncate=False)

Source Data 
+-------+----------+----------+----------+----------+
|PROD_ID|PRICE_2017|PRICE_2018|PRICE_2019|PRICE_2020|
+-------+----------+----------+----------+----------+
| PR0001|       100|       200|       300|       400|
| PR0002|        50|       200|       300|       100|
+-------+----------+----------+----------+----------+

Target Data 
+----------+-----------------+-----------------+-----------------+-----------------+
|PRODUCT_ID|REPEAT_PRICE_2017|REPEAT_PRICE_2018|REPEAT_PRICE_2019|REPEAT_PRICE_2020|
+----------+-----------------+-----------------+-----------------+-----------------+
|PR0001    |100100100        |200200200        |300300300        |400400400        |
|PR0002    |505050           |200200200        |300300300        |100100100        |
+----------+-----------------+-----------------+-----------------+-----------------+



### Compare Same two Data Sets and get the difference only records which are not matching in both the data sets

In [15]:
from pyspark.sql.functions import *

product_2019 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Transformations\Purchase_2019.csv")
)

product_2020 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Transformations\Purchase_2020.csv")
)
print("Source Data - 2019")
product_2019.show()
print("Source Data - 2020")
product_2020.show()

print("Target Data - Difference of 2019 and 2020")
product_2019.exceptAll(product_2020).union(product_2020.exceptAll(product_2019)).show(truncate=False)

Source Data - 2019
+----------+--------------+
|PRODUCT_ID|PURCHASE_PRICE|
+----------+--------------+
|   PRD0001|           100|
|   PRD0002|           200|
|   PRD0003|           300|
|   PRD0004|           400|
+----------+--------------+

Source Data - 2020
+----------+--------------+
|PRODUCT_ID|PURCHASE_PRICE|
+----------+--------------+
|   PRD0001|           100|
|   PRD0002|           200|
|   PRD0003|           300|
|   PRD0004|           350|
|   PRD0005|           500|
+----------+--------------+

Target Data - Difference of 2019 and 2020
+----------+--------------+
|PRODUCT_ID|PURCHASE_PRICE|
+----------+--------------+
|PRD0004   |400           |
|PRD0004   |350           |
|PRD0005   |500           |
+----------+--------------+



### Explode

In [23]:
from pyspark.sql.functions import *
from pyspark.sql.types import ArrayType, StringType

product_arrary_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Transformations\Products_Arrary.csv")
)
print("Source Data ")
product_arrary_df.show(truncate=False)

product_arrary_parsed_df = product_arrary_df.withColumn(
    "Product_Properties_Array",
    from_json(col("Product_Properties"), ArrayType(StringType()))
)

print("Target Data - Exploding Product Properties Array")
product_arrary_parsed_df.select(
    col("Product_id"),
    explode(col("Product_Properties_Array")).alias("Product_Properties")
).show(truncate=False)


Source Data 
+----------+--------------------------+
|Product_Id|Product_Properties        |
+----------+--------------------------+
|PRD0001   |["Black","101 grams","5G"]|
|PRD0002   |["Blue","100 grams","4G"] |
|PRD0003   |["Black","101 grams","5G"]|
|PRD0004   |["Blue","100 grams","3G"] |
+----------+--------------------------+

Target Data - Exploding Product Properties Array
+----------+------------------+
|Product_id|Product_Properties|
+----------+------------------+
|PRD0001   |Black             |
|PRD0001   |101 grams         |
|PRD0001   |5G                |
|PRD0002   |Blue              |
|PRD0002   |100 grams         |
|PRD0002   |4G                |
|PRD0003   |Black             |
|PRD0003   |101 grams         |
|PRD0003   |5G                |
|PRD0004   |Blue              |
|PRD0004   |100 grams         |
|PRD0004   |3G                |
+----------+------------------+

